# Generation and plots

In [ ]:
from pathlib import Path
from tqdm.notebook import tqdm

from combra import data, angles

# --- config (grain types, class order, gen->real class maps) ---
TYPES_DICT = {
    "Ultra_Co11": "мелкие зерна",
    "Ultra_Co25": "средние зерна",
    "Ultra_Co8": "средне-мелкие зерна",
    "Ultra_Co6_2": "крупные зерна",
    "Ultra_Co15": "средне-мелкие зерна",
}

CLASSES = ['Ultra_Co11', 'Ultra_Co25', 'Ultra_Co6_2']

# --- shared source manifest: drives BOTH generation and the plots below ---
# Each source: (path, n_list, family, resolution). Originals are raw image
# folders; san/diffit are h5. n_list = per-class N budgets for that source.
# Defined here (not in the generation cell) so the plot cells run without
# re-running the heavy angle extraction.
# Douglas-Peucker tolerance of combra.angles.vertex_angles (P6, combra >= 0.15);
# it names the sweep folder (_tol1.75). The _msl5 folders beside it hold the
# pre-0.15 P0 extraction, which is not comparable with this one.
ANGLES_TOL = 1.75
OUT_DIR = Path('./data/angles')
SOURCES = [
    ('./datasets/san/o_bc_left_4x_1536_1024x1024_256x256_rgb_N360',   [360],           'real',    256),
    ('./datasets/san/o_bc_left_4x_1536_1024x1024_512x512_rgb_N360',   [360],           'real',    512),
    ('./datasets/san/o_bc_left_4x_1536_1024x1024_1024x1024_rgb_N360', [360],           'real',   1024),
    ('./data/h5/gen_san_256x256_N100_000.h5',                 [1_000, 10_000], 'san',     256),
    ('./data/h5/gen_san_512x512_N100_000.h5',                 [1_000, 10_000], 'san',     512),
    ('./data/h5/00017-diffit-256-gpus2-batch192_N10000.h5',   [1_000, 10_000], 'diffit',  256),
    ('./data/h5/00018-diffit-512-gpus4-batch256_N10000.h5',   [1_000, 10_000], 'diffit',  512),
]

# Generation

In [ ]:
# Unified angle-extraction sweep over the SOURCES manifest (defined in the
# config cell above). sweep_angles skips any (N, steps) parquet already
# on disk unless FORCE=True; family/resolution land in the parquet run_meta.
# Comment a SOURCES row above to skip its (re)generation.
STEPS = [0.1, 0.5, 1, 2, 3, 4, 5]
FORCE = True  # True: rebuild every parquet (and, via force_rebuild_cache, every prep cache)
GEN_KW = dict(workers=20, angles_tol=ANGLES_TOL,
              keep_contours=False, chunksize=64, force_rebuild_cache=FORCE)

for path, n_list, family, resolution in tqdm(SOURCES, desc='sources'):
    out = angles.output_directory(OUT_DIR, path, ANGLES_TOL)
    data.sweep_angles(
        path, out, ns=n_list, step=STEPS, class_types=TYPES_DICT,
        tag=family, force=FORCE,
        run_meta={'family': family, 'resolution': resolution, 'tags': [], 'notes': ''},
        **GEN_KW,
    )
    for n in n_list:
        print(f'[{family}] saved {out / f"angles_n{n}.parquet"}')

# Base plot

In [ ]:
# Single-parquet base plot: N×M grid of per-image angle scatter for one class set.

# path = './data/angles/gen_san_512x512_N100_000_tol1.75/angles_n10000.parquet'
# path = './data/angles/o_bc_left_4x_1536_1024x1024_256x256_rgb_N360_tol1.75/angles_n360.parquet'

path = './data/angles/o_bc_left_4x_1536_1024x1024_512x512_rgb_N360_tol1.75/angles_n360.parquet'

angles.plot_density(parquet_path=path, n_rows=10, n_cols=7, font_size=20, scatter_size=5,
                        step=2, 
                        # ylim=[0, 0.03]
                        )

# Grid of plots

In [ ]:
# Grid plot: orig vs generated (san AND diffit) at each (resolution × grain class).
# Both generators are overlaid on the same axes when their parquets are available.
# Everything (folders, resolutions, per-source N, the compare table) is resolved
# from SOURCES inside plot_overlay_grid, so this cell is just config.

# Generated and real parquets now name their classes identically, so each
# generator's real->gen mapping is the identity.
GEN_NAME_FOR_PER_MODE = {mode: {c: c for c in CLASSES} for mode in ('san', 'diffit')}
STYLES = {
    'orig':   dict(color='blue',   marker='circle'),
    'san':    dict(color='orange', marker='square'),
    'diffit': dict(color='green',  marker='diamond'),
}

# Columns in grain-size order, with their RU labels — from the config above.
grain_classes = [(c, TYPES_DICT[c]) for c in CLASSES]
# Generator families in declaration order (everything in SOURCES that isn't real).
gen_modes = list(dict.fromkeys(fam for _, _, fam, _ in SOURCES if fam != 'real'))

step = 2
save = True
ylim = [0, 0.03]

for max_n in [1_000, 10_000]:
    # compare=True prints the angle-space EMD table (degrees) before each plot.
    angles.plot_overlay_grid(
        OUT_DIR, SOURCES, n=max_n, tol=ANGLES_TOL,
        classes=[k for k, _ in grain_classes],
        col_titles=[label for _, label in grain_classes],
        gen_name_for_mode=GEN_NAME_FOR_PER_MODE, styles=STYLES,
        step=step, ylim=ylim, compare=True,
        title=f'Распределения углов (orig+{"+".join(gen_modes)}, step={step}, N изобр. на класс={max_n})',
        save_path=f'angles_grid_{"_".join(gen_modes)}_n{max_n}_step{step}.png' if save else None,
    )